In [ ]:
import numpy as np
import pandas as pd
from gensim.models import Word2Vec
from gensim.utils import simple_preprocess
from gensim.parsing.preprocessing import STOPWORDS, strip_tags
from sklearn.model_selection import train_test_split
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Input, Embedding, GlobalAveragePooling1D, Dense, Dropout
from tensorflow.keras.utils import pad_sequences

In [ ]:
df = pd.read_csv("IMDB Dataset.csv")
df = df.dropna(subset=["review", "sentiment"]).drop_duplicates(subset="review")
stopwords = STOPWORDS - {"no", "not", "never"}  # Keep sentiment-changing words.
df["words"] = [[word for word in simple_preprocess(strip_tags(review)) if word not in stopwords] for review in df["review"]]
df = df[df["words"].str.len() > 0]
df["label"] = df["sentiment"].map({"negative": 0, "positive": 1})
train, test = train_test_split(df, test_size=0.2, stratify=df["label"], random_state=42)
train, val = train_test_split(train, test_size=0.1, stratify=train["label"], random_state=42)
train_sentences = train["words"].tolist()

In [ ]:
cbow = Word2Vec(train_sentences, vector_size=100, min_count=1, sg=0, epochs=20, workers=1, seed=42)

In [ ]:
# Index 0 is padding; index 1 is an unknown word.
vocab = {word: i + 2 for i, word in enumerate(cbow.wv.index_to_key)}
vectors = np.vstack([np.zeros((2, 100), dtype="float32"), cbow.wv.vectors])
length = min(300, max(len(words) for words in train_sentences))
sequences = []
for frame in [train, val, test]:
    ids = [[vocab.get(word, 1) for word in words] for words in frame["words"]]
    sequences.append(pad_sequences(ids, maxlen=length, padding="post", truncating="post"))
x_train, x_val, x_test = sequences
y_train = train["label"].to_numpy()
y_val = val["label"].to_numpy()
y_test = test["label"].to_numpy()

In [ ]:
model = Sequential([
    Input((length,), dtype="int32"),
    Embedding(len(vectors), 100, weights=[vectors], trainable=False, mask_zero=True),
    GlobalAveragePooling1D(),
    Dense(64, activation="relu"), Dropout(0.2), Dense(1, activation="sigmoid")])
model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
model.fit(x_train, y_train, epochs=20, batch_size=128, validation_data=(x_val, y_val))

In [ ]:
loss, accuracy = model.evaluate(x_test, y_test)
print("Test loss:", loss)
print("Test accuracy (%):", round(100 * accuracy, 2))